# Customer Segmentation & Targeting Strategy

**Author:** Raaga Priya JK  
**Dataset:** Customer Personality Analysis (Kaggle)  
**Techniques:** K-Means++, DBSCAN, Hierarchical Clustering, PCA

---

## Objective
Segment 2,212 customers into actionable groups based on demographics, spending behaviour, and campaign responsiveness to drive targeted marketing strategies.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261', '#264653']
print('Libraries loaded successfully')

## 2. Load & Inspect Data

In [ ]:
try:
    df = pd.read_csv('../data/marketing_campaign.csv', sep='\t')
    if df.shape[1] < 5:
        df = pd.read_csv('../data/marketing_campaign.csv')
except:
    df = pd.read_csv('../data/marketing_campaign.csv')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nData types:')
print(df.dtypes)

## 3. Data Cleaning

In [ ]:
before = len(df)

# Drop useless columns
drop_cols = ['Z_CostContact', 'Z_Revenue', 'ID']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# Drop missing Income
df.dropna(subset=['Income'], inplace=True)

# Remove outliers
df = df[df['Income'] < 200000]
df = df[df['Year_Birth'] >= 1930]

# Fix Marital Status
df['Marital_Status'] = df['Marital_Status'].replace(
    {'Absurd': 'Single', 'YOLO': 'Single', 'Alone': 'Single'}
)

print(f'Rows before: {before} | Rows after: {len(df)} | Removed: {before - len(df)}')
print(f'Unique Marital Status: {df["Marital_Status"].unique()}')

## 4. Feature Engineering

In [ ]:
df['Age']              = 2024 - df['Year_Birth']
df['TotalSpend']       = (df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] +
                          df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds'])
df['TotalPurchases']   = (df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases'])
df['TotalChildren']    = df['Kidhome'] + df['Teenhome']
df['CampaignAccepted'] = df[['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3',
                              'AcceptedCmp4','AcceptedCmp5']].sum(axis=1)
df['SpendPerPurchase'] = np.where(df['TotalPurchases'] > 0,
                                   df['TotalSpend'] / df['TotalPurchases'], 0)

edu_map = {'Basic':0,'2n Cycle':1,'Graduation':2,'Master':3,'PhD':4}
df['Edu_Encoded'] = df['Education'].map(edu_map)

print('Engineered features:')
df[['Age','TotalSpend','TotalPurchases','TotalChildren','CampaignAccepted']].describe().round(2)

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Customer Personality Analysis - EDA', fontsize=15, fontweight='bold')

axes[0,0].hist(df['Income'], bins=40, color='#457B9D', edgecolor='white', lw=0.5)
axes[0,0].set_title('Income Distribution', fontweight='bold')
axes[0,0].set_xlabel('Annual Income ($)')

axes[0,1].hist(df['TotalSpend'], bins=40, color='#2A9D8F', edgecolor='white', lw=0.5)
axes[0,1].set_title('Total Spend Distribution', fontweight='bold')
axes[0,1].set_xlabel('Total Spend ($)')

spend_cols = ['MntWines','MntMeatProducts','MntFishProducts','MntFruits','MntSweetProducts','MntGoldProds']
spend_means = df[spend_cols].mean().sort_values()
axes[0,2].barh(spend_means.index, spend_means.values, color='#E9C46A')
axes[0,2].set_title('Avg Spend by Category', fontweight='bold')

sc = axes[1,0].scatter(df['Income'], df['TotalSpend'], alpha=0.3,
                        c=df['Age'], cmap='plasma', s=15)
axes[1,0].set_title('Income vs Total Spend (by Age)', fontweight='bold')
plt.colorbar(sc, ax=axes[1,0], label='Age')

camp_total = df[['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5','Response']].sum()
axes[1,1].bar(camp_total.index, camp_total.values, color=PALETTE, width=0.6)
axes[1,1].set_title('Campaign Acceptance Count', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=30)

edu_spend = df.groupby('Education')['TotalSpend'].mean().sort_values()
axes[1,2].barh(edu_spend.index, edu_spend.values, color='#F4A261')
axes[1,2].set_title('Avg Spend by Education', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr_cols = ['Income','Age','TotalSpend','TotalPurchases','TotalChildren','CampaignAccepted','Recency']
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Preprocessing for Clustering

In [ ]:
cluster_features = ['Income','Age','TotalSpend','TotalPurchases',
                    'TotalChildren','CampaignAccepted','Recency','Edu_Encoded']
X = df[cluster_features].fillna(df[cluster_features].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Feature matrix shape: {X_scaled.shape}')

## 7. Optimal K - Elbow & Silhouette

In [ ]:
inertias, sil_scores = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=15, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertias, 'o-', color='#457B9D', lw=2, ms=7)
axes[0].axvline(x=4, color='#E63946', ls='--', alpha=0.7, label='Optimal K=4')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].legend()

best_k = list(K_range)[sil_scores.index(max(sil_scores))]
axes[1].plot(list(K_range), sil_scores, 's-', color='#2A9D8F', lw=2, ms=7)
axes[1].axvline(x=best_k, color='#E63946', ls='--', alpha=0.7, label=f'Best K={best_k}')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Best K by silhouette: {best_k} | Score: {max(sil_scores):.4f}')

## 8. K-Means++ Clustering

In [ ]:
kmeans = KMeans(n_clusters=4, init='k-means++', n_init=20, random_state=42)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df['KMeans_Cluster'])
print(f'K-Means Silhouette Score: {sil:.4f}')
print(f'Cluster sizes:\n{df["KMeans_Cluster"].value_counts().sort_index()}')

## 9. DBSCAN Clustering

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=10)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

n_clusters = len(set(df['DBSCAN_Cluster'])) - (1 if -1 in df['DBSCAN_Cluster'].values else 0)
n_noise    = (df['DBSCAN_Cluster'] == -1).sum()

valid_mask = df['DBSCAN_Cluster'] != -1
if n_clusters > 1:
    sil_db = silhouette_score(X_scaled[valid_mask], df.loc[valid_mask, 'DBSCAN_Cluster'])
    print(f'DBSCAN Clusters found: {n_clusters}')
    print(f'Noise points: {n_noise} ({n_noise/len(df)*100:.1f}%)')
    print(f'DBSCAN Silhouette Score: {sil_db:.4f}')
else:
    print(f'DBSCAN found {n_clusters} cluster — try tuning eps/min_samples')
    print(f'Noise points: {n_noise}')

## 10. Hierarchical Clustering

In [ ]:
# Dendrogram on a sample
sample_idx = np.random.choice(len(X_scaled), 200, replace=False)
X_sample   = X_scaled[sample_idx]

linked = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(linked, ax=ax, truncate_mode='lastp', p=20,
           leaf_rotation=45, leaf_font_size=10, color_threshold=10)
ax.set_title('Hierarchical Clustering Dendrogram (sample of 200)', fontweight='bold')
ax.set_xlabel('Customer Index')
ax.set_ylabel('Ward Distance')
plt.tight_layout()
plt.show()

In [ ]:
# Fit Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
df['Hierarchical_Cluster'] = agg.fit_predict(X_scaled)

sil_agg = silhouette_score(X_scaled, df['Hierarchical_Cluster'])
print(f'Hierarchical Silhouette Score: {sil_agg:.4f}')
print(f'Cluster sizes:\n{df["Hierarchical_Cluster"].value_counts().sort_index()}')

## 11. Model Comparison

In [ ]:
sil_km  = silhouette_score(X_scaled, df['KMeans_Cluster'])
sil_agg = silhouette_score(X_scaled, df['Hierarchical_Cluster'])

valid_mask = df['DBSCAN_Cluster'] != -1
n_db = len(set(df['DBSCAN_Cluster'])) - 1
sil_db = silhouette_score(X_scaled[valid_mask], df.loc[valid_mask,'DBSCAN_Cluster']) if n_db > 1 else 0

comparison = pd.DataFrame({
    'Algorithm':         ['K-Means++', 'Hierarchical (Ward)', 'DBSCAN'],
    'Clusters':          [4, 4, n_db],
    'Silhouette Score':  [round(sil_km,4), round(sil_agg,4), round(sil_db,4)],
    'Noise Points':      ['None', 'None', str(n_noise)],
    'Best For':          ['Scalable segmentation', 'Hierarchical relationships', 'Outlier detection']
})

print('Algorithm Comparison:')
print(comparison.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
algos  = ['K-Means++', 'Hierarchical', 'DBSCAN']
scores = [sil_km, sil_agg, sil_db]
colors = ['#457B9D', '#2A9D8F', '#E9C46A']
bars = ax.bar(algos, scores, color=colors, width=0.4, edgecolor='white')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{score:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Silhouette Score Comparison Across Algorithms', fontweight='bold')
ax.set_ylabel('Silhouette Score')
ax.set_ylim(0, max(scores) * 1.3)
plt.tight_layout()
plt.show()

## 12. Final Cluster Analysis (K-Means++)

In [ ]:
# PCA visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'], df['PCA2'] = X_pca[:,0], X_pca[:,1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('K-Means++ Segmentation Results', fontsize=14, fontweight='bold')

SEGMENT_NAMES = {0:'Budget Families', 1:'Consistent Low-Spenders',
                 2:'Premium High-Value', 3:'Campaign Responsive'}

for c in range(4):
    mask = df['KMeans_Cluster'] == c
    axes[0].scatter(df.loc[mask,'PCA1'], df.loc[mask,'PCA2'],
                    c=PALETTE[c], label=f'C{c}: {SEGMENT_NAMES[c]}',
                    alpha=0.5, s=20)
centres_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centres_pca[:,0], centres_pca[:,1], c='black',
                marker='X', s=200, zorder=5, label='Centroids')
axes[0].set_title('PCA Projection', fontweight='bold')
axes[0].legend(fontsize=8)

for c in range(4):
    mask = df['KMeans_Cluster'] == c
    axes[1].scatter(df.loc[mask,'Income'], df.loc[mask,'TotalSpend'],
                    c=PALETTE[c], label=f'C{c}', alpha=0.5, s=20)
axes[1].set_title('Income vs Total Spend', fontweight='bold')
axes[1].set_xlabel('Annual Income ($)')
axes[1].set_ylabel('Total Spend ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Cluster Profiles & Business Insights

In [ ]:
profile_cols = ['Income','Age','TotalSpend','TotalPurchases','TotalChildren','CampaignAccepted','Recency']
profile = df.groupby('KMeans_Cluster')[profile_cols].mean().round(1)
profile['Count']   = df['KMeans_Cluster'].value_counts().sort_index()
profile['Segment'] = [SEGMENT_NAMES[i] for i in profile.index]
print(profile.to_string())

In [ ]:
insights = {
    'Premium High-Value':      'Loyalty rewards, early access, premium upsell',
    'Campaign Responsive':     'Personalised offers, discount triggers, retargeting',
    'Budget Families':         'Value bundles, family deals, BNPL offers',
    'Consistent Low-Spenders': 'Re-engagement emails, time-limited incentives'
}
print('Business Recommendations:')
print('-' * 65)
for seg, rec in insights.items():
    print(f'  {seg:28s}: {rec}')

---
## Summary

| Metric | Value |
|---|---|
| Customers Segmented | 2,212 |
| Best Algorithm | K-Means++ |
| Optimal Clusters | 4 |
| Silhouette Score | 0.1838 (acceptable for overlapping real-world data) |
| PCA Explained Variance | ~53% |

**K-Means++ was selected as the final model** due to its balance of performance, scalability, and interpretability compared to Hierarchical and DBSCAN approaches.